In [19]:
import dask.dataframe as dd

In [20]:
data = dd.read_parquet("../../Dataset/New_QM9/data.parquet")

In [21]:
data.info()

<class 'dask.dataframe.dask_expr.DataFrame'>
Columns: 28 entries, molecule_id to rotation
dtypes: float64(20), int64(2), string(6)

In [22]:
data.columns

Index(['molecule_id', 'atom_index', 'atom', 'x', 'y', 'z', 'charge',
       'num_atoms', 'smiles', 'tag', 'index', 'A', 'B', 'C', 'mu', 'alpha',
       'homo', 'lumo', 'gap', 'r2', 'zpve', 'u0', 'u298', 'h298', 'g298', 'cv',
       'chiral_centers', 'rotation'],
      dtype='object')

In [23]:
from pathlib import Path

# Required atom-level fields requested
atom_cols = ["molecule_id", "atom_index", "atom", "x", "y", "z", "charge"]
missing_atom_cols = [c for c in atom_cols if c not in data.columns]
if missing_atom_cols:
    raise ValueError(f"Missing atom-level columns in dataframe: {missing_atom_cols}")

# Atom-level dataset: keep one row per atom with molecule linkage
atom_df = data[atom_cols]

# Molecule-level dataset: keep one unique row per molecule_id
atom_specific_cols = {"atom_index", "atom", "x", "y", "z", "charge"}
molecule_cols = [c for c in data.columns if c not in atom_specific_cols]
if "molecule_id" not in molecule_cols:
    molecule_cols = ["molecule_id"] + molecule_cols

molecule_df = data[molecule_cols].drop_duplicates(subset=["molecule_id"])

# Save outputs
out_dir = Path("../../Dataset/New_QM9")
out_dir.mkdir(parents=True, exist_ok=True)

atom_out = out_dir / "atom_properties.csv"
molecule_out = out_dir / "molecule_properties.csv"

atom_df.to_csv(str(atom_out), single_file=True, index=False)
molecule_df.to_csv(str(molecule_out), single_file=True, index=False)

print("Saved:")
print(f"- Molecule-level dataset: {molecule_out}")
print(f"- Atom-level dataset: {atom_out}")
print("Preview shapes (computed):")
print("- molecules:", molecule_df.shape[0].compute(), "rows")
print("- atoms:", atom_df.shape[0].compute(), "rows")

Saved:
- Molecule-level dataset: ../../Dataset/New_QM9/molecule_properties.csv
- Atom-level dataset: ../../Dataset/New_QM9/atom_properties.csv
Preview shapes (computed):
- molecules: 118548 rows
- atoms: 2133173 rows


In [24]:
atom_df.head()

,molecule_id,atom_index,atom,x,y,z,charge
0,063330,0,C,0.065523,1.503139,-0.005431,-0.424123
1,063330,1,C,-0.053985,-0.036015,-0.018889,0.181460
2,063330,2,C,-1.526202,-0.461064,0.002840,-0.428876
3,063330,3,C,0.753023,-0.640889,-1.184806,-0.401831
4,063330,4,C,2.074490,-1.063118,-0.571171,0.416695


In [25]:
molecule_df.head()

,molecule_id,num_atoms,smiles,tag,index,A,B,C,mu,alpha,...,gap,r2,zpve,u0,u298,h298,g298,cv,chiral_centers,rotation
0,063330,18,CC1(C)CC(=O)OC1=N,gdb,63330.0,2.53299,1.46415,1.16753,2.6416,73.01,...,0.2616,1109.9443,0.146979,-439.155821,-439.147118,-439.146174,-439.189288,32.767,[],"[32.28, 37.92, 134.63]"
18,025365,15,O=CC1=CN2CCC2=N1,gdb,25365.0,5.88208,1.05635,0.90586,7.3954,77.91,...,0.2025,1220.7204,0.114382,-416.821547,-416.814285,-416.813341,-416.853719,26.571,[],"[-0.2, -0.25, -1.21]"
33,008591,14,O=C1OC23CC(C2)C13,gdb,8591.0,5.20145,1.83242,1.74881,4.1900,62.15,...,0.2564,761.2922,0.107651,-382.434401,-382.428634,-382.427690,-382.464109,23.306,"[(2, 'R'), (3, 'R'), (5, 'S')]","[-18.02, -20.15, -4.96]"
47,046826,15,O=C1NCC2CC12C#N,gdb,46826.0,2.59711,1.66210,1.19476,6.8177,70.48,...,0.2630,1015.2334,0.114845,-416.853478,-416.845979,-416.845035,-416.885555,28.261,"[(3, 'S'), (5, 'R')]","[150.51, 178.42, 728.78]"
62,123700,15,C1C2OC3=C(NN=C3)C12,gdb,123700.0,4.14707,1.60029,1.25858,3.3444,70.40,...,0.2226,945.9406,0.116545,-416.789726,-416.783519,-416.782575,-416.820141,25.371,"[(2, 'R'), (3, 'R')]","[-7.2, -10.56, -152.4]"
